## Streamlined Building Unit Cleaning Code, for San Diego County

Used for joining with SDGE utility

In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [3]:
# load parcels only for San Diego county (will take about 5 minutes)
parcels = gpd.read_file(
   "/../../capstone/electrigrid/data/parcels/Parcels_CA_2014.gdb",
    layer="CA_PARCELS_STATEWIDE",
    where="County='San Diego'").to_crs(epsg=4326)

In [4]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [5]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

## Streamlined Code 

In [6]:
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

In [7]:

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

In [8]:

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

building_zillow = building_zillow.drop(columns = "index_right")

The problem I think that's happening here is that the units are added to every single building. When multiple buildings are in one parcel the units are double counted (?). So we need to add in the parcel number and either aggregate to the parcel or divide by the number of buildings. It's probably easiest to aggregate the areas to parcel so that we don't get fractional numbers of homes. 

In [9]:
# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels,
                                how = "left",
                                predicate = "intersects")

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'ADDRESS', 'CITY', 'ZIP','Shape_Length', 'Shape_Area'])

In [10]:
building_zillow_parcels

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-116.98014 32.61383, -116.98015 32.6...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004100,San Diego
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-116.98014 32.61383, -116.98015 32.6...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004200,San Diego
7717377,osm,1009026928,10.977699,6.832924,USA,"{'xmax': -116.96362569999997, 'xmin': -116.964...","POLYGON ((-116.96464 32.61837, -116.96376 32.6...",Multi,2018.0,NaN,None,None,None,123.0,10388986.0,living,73407.0,8414651,06073013319,h493,SDGE,RI104,None,San Diego
7718471,ms,UnitedStates_023013221_510206,5.247350,3.380768,USA,"{'xmax': -116.96742523067107, 'xmin': -116.967...","POLYGON ((-116.96783 32.61175, -116.96760 32.6...",Multi,2021.0,NaN,None,None,None,253.0,114957235.0,None,NaN,8435506,06073013319,h493,SDGE,RI104,None,San Diego


In [11]:
# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

In [12]:
building_m

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004100,San Diego
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004200,San Diego
7717377,osm,1009026928,10.977699,6.832924,USA,"{'xmax': -116.96362569999997, 'xmin': -116.964...","POLYGON ((-11285483.179 3945189.515, -11285398...",Multi,2018.0,NaN,None,None,None,123.0,10388986.0,living,73407.0,8414651,06073013319,h493,SDGE,RI104,None,San Diego
7718471,ms,UnitedStates_023013221_510206,5.247350,3.380768,USA,"{'xmax': -116.96742523067107, 'xmin': -116.967...","POLYGON ((-11285790.455 3944475.178, -11285768...",Multi,2021.0,NaN,None,None,None,253.0,114957235.0,None,NaN,8435506,06073013319,h493,SDGE,RI104,None,San Diego


In [13]:
# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

In [14]:
building_m

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004100,San Diego,196.444060,926.344867
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004200,San Diego,196.444060,926.344867
7717377,osm,1009026928,10.977699,6.832924,USA,"{'xmax': -116.96362569999997, 'xmin': -116.964...","POLYGON ((-11285483.179 3945189.515, -11285398...",Multi,2018.0,NaN,None,None,None,123.0,10388986.0,living,73407.0,8414651,06073013319,h493,SDGE,RI104,None,San Diego,3435.935102,37718.662300
7718471,ms,UnitedStates_023013221_510206,5.247350,3.380768,USA,"{'xmax': -116.96742523067107, 'xmin': -116.967...","POLYGON ((-11285790.455 3944475.178, -11285768...",Multi,2021.0,NaN,None,None,None,253.0,114957235.0,None,NaN,8435506,06073013319,h493,SDGE,RI104,None,San Diego,526.185918,2761.081538


In [15]:
# keep only observations with unit data
building_w_units = building_m[~building_m['unit'].isna()]

assert building_w_units['unit'].isna().sum() == 0

Aggregate the volumes for the unit regression by parcel.

In [16]:
# aggregate the volumes for the unit regression by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# add the value into the dataframe to keep all of the other data 

In [17]:
total_volume_m3

PARNO
1026304400    1175.703563
1026305000    2209.766323
1026701900    1426.100866
1026702000    1426.100866
1030670900    2429.569869
                 ...     
7766019127     254.077998
7766019128     254.077998
7766019129     254.077998
7766019130     254.077998
7766019131     254.077998
Name: volume_m3, Length: 43258, dtype: float64

In [18]:
# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

In [19]:
# some of the homes don't have a parcel number so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

In [21]:
building_w_units

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915,905.530915
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436,1177.213436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7715428,osm,1120169118,9.802270,3.607025,USA,"{'xmax': -116.98862620000001, 'xmin': -116.989...","POLYGON ((-11287835.939 3944949.530, -11287830...",Multi,2015.0,NaN,None,None,None,187.0,50592225.0,None,NaN,8400963,06073013317,h493,SDGE,RI104,6443101000,San Diego,903.386384,8855.237190,8855.237190
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004100,San Diego,196.444060,926.344867,926.344867
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004200,San Diego,196.444060,926.344867,926.344867
7717377,osm,1009026928,10.977699,6.832924,USA,"{'xmax': -116.96362569999997, 'xmin': -116.964...","POLYGON ((-11285483.179 3945189.515, -11285398...",Multi,2018.0,NaN,None,None,None,123.0,10388986.0,living,73407.0,8414651,06073013319,h493,SDGE,RI104,None,San Diego,3435.935102,37718.662300,37718.662300


In [22]:
# run model
results = smf.ols('unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

In [23]:
building_w_units

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3,residual
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-11288598.440 4018349.084, -11288592...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247,1606.752247,-31.544531
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-11287597.377 4013374.612, -11287563...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171,1858.907171,-31.532753
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915,905.530915,-31.577284
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-11228263.448 4013304.415, -11228255...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915,905.530915,-31.577284
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-11228325.006 4013391.447, -11228324...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436,1177.213436,-31.564594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7715428,osm,1120169118,9.802270,3.607025,USA,"{'xmax': -116.98862620000001, 'xmin': -116.989...","POLYGON ((-11287835.939 3944949.530, -11287830...",Multi,2015.0,NaN,None,None,None,187.0,50592225.0,None,NaN,8400963,06073013317,h493,SDGE,RI104,6443101000,San Diego,903.386384,8855.237190,8855.237190,153.794042
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004100,San Diego,196.444060,926.344867,926.344867,-31.576312
7715815,osm,1120565410,4.715566,0.911623,USA,"{'xmax': -116.97998779999999, 'xmin': -116.980...","POLYGON ((-11286978.360 3944699.625, -11286979...",Multi,2010.0,5.0,None,None,O,2.0,751590.0,living,2992.0,8336605,06073013317,h493,SDGE,RI101,6443004200,San Diego,196.444060,926.344867,926.344867,-31.576312
7717377,osm,1009026928,10.977699,6.832924,USA,"{'xmax': -116.96362569999997, 'xmin': -116.964...","POLYGON ((-11285483.179 3945189.515, -11285398...",Multi,2018.0,NaN,None,None,None,123.0,10388986.0,living,73407.0,8414651,06073013319,h493,SDGE,RI104,None,San Diego,3435.935102,37718.662300,37718.662300,91.142238


In [24]:
# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

In [25]:
building_units_clean.shape

(48541, 28)

In [26]:
building_outliers.shape

(3122, 28)

In [27]:
# rerun linear regression
results_clean = smf.ols('unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

/tmp/ipykernel_1879087/26689504.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_1879087/26689504.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [28]:
intercept

17.100973596715377

In [29]:
slope

5.598000083463197e-06

In [30]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[building_m['unit'].isna()]

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['unit'] = round(intercept + missing_outlier_units_pred['volume_m3'] * slope)

In [35]:
missing_units

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3
6512306,osm,1286587076,3.777157,0.320945,USA,"{'xmax': -117.3696614, 'xmin': -117.3697478, '...","POLYGON ((-11324567.948 4005781.017, -11324570...",Multi,NaN,NaN,None,None,None,NaN,83106.0,None,NaN,7509790,06073018201,h511,SDGE,RI109,1521730400,San Diego,99.033027,374.063251
6519495,osm,1286623786,4.366043,0.966425,USA,"{'xmax': -117.38648509999999, 'xmin': -117.386...","POLYGON ((-11326190.287 4008034.672, -11326185...",Multi,NaN,NaN,None,None,None,NaN,859315.0,None,NaN,7500738,06073018400,h511,SDGE,RI109,7714304224,San Diego,101.217850,441.921448
6519495,osm,1286623786,4.366043,0.966425,USA,"{'xmax': -117.38648509999999, 'xmin': -117.386...","POLYGON ((-11326190.287 4008034.672, -11326185...",Multi,NaN,NaN,None,None,None,NaN,859315.0,None,NaN,7500738,06073018400,h511,SDGE,RI109,7714304208,San Diego,101.217850,441.921448
6519495,osm,1286623786,4.366043,0.966425,USA,"{'xmax': -117.38648509999999, 'xmin': -117.386...","POLYGON ((-11326190.287 4008034.672, -11326185...",Multi,NaN,NaN,None,None,None,NaN,859315.0,None,NaN,7500738,06073018400,h511,SDGE,RI109,7714304209,San Diego,101.217850,441.921448
6519495,osm,1286623786,4.366043,0.966425,USA,"{'xmax': -117.38648509999999, 'xmin': -117.386...","POLYGON ((-11326190.287 4008034.672, -11326185...",Multi,NaN,NaN,None,None,None,NaN,859315.0,None,NaN,7500738,06073018400,h511,SDGE,RI109,7714304212,San Diego,101.217850,441.921448
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7682630,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-11285404.601 3957048.806, -11285403...",Multi,NaN,NaN,None,None,None,NaN,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523531,San Diego,127.948589,657.903862
7682630,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-11285404.601 3957048.806, -11285403...",Multi,NaN,NaN,None,None,None,NaN,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523406,San Diego,127.948589,657.903862
7682630,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-11285404.601 3957048.806, -11285403...",Multi,NaN,NaN,None,None,None,NaN,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523359,San Diego,127.948589,657.903862
7682630,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-11285404.601 3957048.806, -11285403...",Multi,NaN,NaN,None,None,None,NaN,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523567,San Diego,127.948589,657.903862


In [ ]:
missing_outlier_units

In [31]:
missing_outlier_units_pred['unit'].agg(['min','max'])

min    17.0
max    19.0
Name: unit, dtype: float64

In [32]:
# combine multi-family homes data frames
multi_complete = pd.concat([building_w_units, missing_outlier_units_pred]).to_crs(zillow.crs)

# drop residual column
multi_complete = multi_complete.drop(['residual'], axis = 1)

multi_by_parcel = parcels_res.sjoin(multi_complete, predicate="intersects")


**DECIDE WHICH UNIT COLUMN TO KEEP** `unit_left` or `unit_right`?

In [33]:
multi_complete

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO,County,area_m2,volume_m3,total_volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,1323607700,San Diego,412.421351,1606.752247,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,1881920200,San Diego,452.260448,1858.907171,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612100,San Diego,232.343850,905.530915,905.530915
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,1980612000,San Diego,232.343850,905.530915,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,1980522300,San Diego,284.844083,1177.213436,1177.213436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6760,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-116.96383 32.72838, -116.96381 32.7...",Multi,NaN,NaN,None,None,None,17.0,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523531,San Diego,127.948589,657.903862,NaN
6761,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-116.96383 32.72838, -116.96381 32.7...",Multi,NaN,NaN,None,None,None,17.0,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523406,San Diego,127.948589,657.903862,NaN
6762,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-116.96383 32.72838, -116.96381 32.7...",Multi,NaN,NaN,None,None,None,17.0,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523359,San Diego,127.948589,657.903862,NaN
6763,osm,1134139161,5.141939,0.867374,USA,"{'xmax': -116.96372869999999, 'xmin': -116.963...","POLYGON ((-116.96383 32.72838, -116.96381 32.7...",Multi,NaN,NaN,None,None,None,17.0,2828071.0,None,NaN,8145055,06073013506,h366,SDGE,RI109,7750523567,San Diego,127.948589,657.903862,NaN


In [34]:
multi_by_parcel

,PARNO_left,County_left,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,PARNO_right,County_right,area_m2,volume_m3,total_volume_m3
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,4236922104,San Diego,140.235708,431.200368,431.200368
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,4236922102,San Diego,140.235708,431.200368,431.200368
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,4236922103,San Diego,140.235708,431.200368,431.200368
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,4236922101,San Diego,140.235708,431.200368,431.200368
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,4236921300,San Diego,140.235708,431.200368,431.200368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040121,2181606400,San Diego,None,None,None,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",2.0,6607446,ms,UnitedStates_023013203_67117,8.271330,2.361229,USA,"{'xmax': -117.1550661828501, 'xmin': -117.1553...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101,2181608006,San Diego,311.556197,2576.984084,2576.984084
1040121,2181606400,San Diego,None,None,None,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",2.0,6607446,ms,UnitedStates_023013203_67117,8.271330,2.361229,USA,"{'xmax': -117.1550661828501, 'xmin': -117.1553...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101,2181608002,San Diego,311.556197,2576.984084,2576.984084
1040121,2181606400,San Diego,None,None,None,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",2.0,6607446,ms,UnitedStates_023013203_67117,8.271330,2.361229,USA,"{'xmax': -117.1550661828501, 'xmin': -117.1553...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101,2181608018,San Diego,311.556197,2576.984084,2576.984084
1040121,2181606400,San Diego,None,None,None,152.632191,1299.475183,"MULTIPOLYGON Z (((-117.15515 33.15440 0.00000,...",2.0,6607446,ms,UnitedStates_023013203_67117,8.271330,2.361229,USA,"{'xmax': -117.1550661828501, 'xmin': -117.1553...",Multi,1976.0,6.0,None,None,None,2.0,290793.0,living,2048.0,7653376,06073020041,h495,SDGE,RI101,2181608025,San Diego,311.556197,2576.984084,2576.984084


In [75]:
# the two unit columns don't match this makes sense since the one is outliers and projected
multi_by_parcel['unit_left'].equals(multi_by_parcel['unit_right'])

False

In [39]:
# check how many parcel numbers are duplicates
duplicates = multi_by_parcel.pivot_table(index = ['PARNO'], aggfunc = 'size')
print(f"There are ",duplicates[duplicates > 1].sum(), " duplicated parcels.")

There are  29066  duplicated parcels.


In [40]:
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

In [ ]:
## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid


In [42]:
multi_by_parcel

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,centroids
151,4236921300,San Diego,None,None,None,69.352091,238.641330,MULTIPOLYGON Z (((-11313159.182 3961251.281 0....,2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,140.235708,431.200368,POINT (-11313153.723 3961239.229)
154,4236220200,San Diego,None,None,None,67.240870,224.046738,MULTIPOLYGON Z (((-11313244.013 3962367.862 0....,3.0,7349569,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,117.733224,320.096607,POINT (-11313238.717 3962356.234)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,4.0,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-11311395.134 3963841.243)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,MULTIPOLYGON Z (((-11311377.586 3963837.438 0....,4.0,7316915,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,224.418227,1755.586108,POINT (-11311395.134 3963841.243)
160,4245420700,San Diego,None,None,None,92.385658,398.294157,MULTIPOLYGON Z (((-11311356.480 3963757.406 0....,2.0,7316886,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,113.884017,734.439912,POINT (-11311373.489 3963745.624)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039825,2632430800,San Diego,None,None,None,137.363218,918.420109,MULTIPOLYGON Z (((-11314226.300 3985891.446 0....,2.0,7367690,ms,UnitedStates_023013221_216529,3.732555,0.827698,USA,"{'xmax': -117.26281072873077, 'xmin': -117.263...",Multi,1969.0,4.0,None,None,O,2.0,1319921.0,living,3047.0,7736193,06073017303,h510,SDGE,RI101,301.800899,1126.488574,POINT (-11314255.652 3985894.577)
1039961,5032808000,San Diego,None,None,None,114.021685,694.906225,MULTIPOLYGON Z (((-11290124.510 3957832.406 0....,2.0,7218469,ms,UnitedStates_023013221_556001,4.677042,0.605590,USA,"{'xmax': -117.01293428361708, 'xmin': -117.013...",Multi,1993.0,6.0,None,None,None,2.0,252491.0,living,2568.0,8141863,06073013802,h302,SDGE,RI110,239.438899,1119.865903,POINT (-11290144.704 3957823.677)
1040095,5182013023,San Diego,None,None,None,784.819873,13648.161643,MULTIPOLYGON Z (((-11281516.606 3959472.038 0....,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)
1040096,5182013025,San Diego,None,None,None,784.819873,13648.161643,MULTIPOLYGON Z (((-11281516.606 3959472.038 0....,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)


In [43]:

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('centroids')


In [63]:
multi_by_parcel_processed

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,0
151,4236921300,San Diego,None,None,None,69.352091,238.641330,2.0,7349122,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,140.235708,431.200368,POINT (-11313153.723 3961239.229)
154,4236220200,San Diego,None,None,None,67.240870,224.046738,3.0,7349569,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,117.733224,320.096607,POINT (-11313238.717 3962356.234)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-11311395.134 3963841.243)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316916,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-11311395.134 3963841.243)
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316915,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,224.418227,1755.586108,POINT (-11311395.134 3963841.243)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039825,2632430800,San Diego,None,None,None,137.363218,918.420109,2.0,7367690,ms,UnitedStates_023013221_216529,3.732555,0.827698,USA,"{'xmax': -117.26281072873077, 'xmin': -117.263...",Multi,1969.0,4.0,None,None,O,2.0,1319921.0,living,3047.0,7736193,06073017303,h510,SDGE,RI101,301.800899,1126.488574,POINT (-11314255.652 3985894.577)
1039961,5032808000,San Diego,None,None,None,114.021685,694.906225,2.0,7218469,ms,UnitedStates_023013221_556001,4.677042,0.605590,USA,"{'xmax': -117.01293428361708, 'xmin': -117.013...",Multi,1993.0,6.0,None,None,None,2.0,252491.0,living,2568.0,8141863,06073013802,h302,SDGE,RI110,239.438899,1119.865903,POINT (-11290144.704 3957823.677)
1040095,5182013023,San Diego,None,None,None,784.819873,13648.161643,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)
1040096,5182013025,San Diego,None,None,None,784.819873,13648.161643,1.0,7521513,osm,1085544737,7.041645,4.969846,USA,"{'xmax': -116.92388439999999, 'xmin': -116.924...",Multi,1986.0,3.0,None,None,O,1.0,520014.0,living,1294.0,8157714,06073013604,h366,SDGE,RI101,237.086541,1669.479270,POINT (-11281579.243 3959538.970)


In [44]:
# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

# subset for single family zillow and condos
non_multi_zillow = zillow[zillow['type'] != "Multi"].to_crs(zillow.crs)

# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, non_multi_zillow], axis = 0)

In [45]:
complete_zillow

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,unit_left,index_right,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,unit_right,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3,centroids,unit,geometry
151,4236921300,San Diego,None,None,None,69.352091,238.641330,2.0,7349122.0,osm,1308122121,3.074826,0.141122,USA,"{'xmax': -117.25140540000001, 'xmin': -117.251...",Multi,1975.0,4.0,None,None,None,2.0,1026707.0,None,NaN,7996232,06073007602,h567,SDGE,RI101,140.235708,431.200368,POINT (-117.25142 32.76728),NaN,None
154,4236220200,San Diego,None,None,None,67.240870,224.046738,3.0,7349569.0,ms,UnitedStates_023013221_128769,2.718830,0.218748,USA,"{'xmax': -117.2521855055026, 'xmin': -117.2522...",Multi,1980.0,5.0,None,None,None,2.0,986647.0,living,1932.0,7995754,06073007601,h530,SDGE,RI101,117.733224,320.096607,POINT (-117.25230 32.77765),NaN,None
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316916.0,ms,UnitedStates_023013221_606679,7.757454,4.108285,USA,"{'xmax': -117.23299227742604, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,160936.0,None,NaN,7998883,06073007702,h530,SDGE,RI110,217.808548,1689.639778,POINT (-117.23320 32.79144),NaN,None
159,4245420100,San Diego,None,None,None,94.646744,497.462992,4.0,7316915.0,ms,UnitedStates_023013221_369659,7.822832,2.910877,USA,"{'xmax': -117.23303161924132, 'xmin': -117.233...",Multi,NaN,NaN,None,None,None,4.0,158049.0,None,NaN,7998882,06073007702,h530,SDGE,RI110,224.418227,1755.586108,POINT (-117.23320 32.79144),NaN,None
160,4245420700,San Diego,None,None,None,92.385658,398.294157,2.0,7316886.0,ms,UnitedStates_023013221_13570,6.449017,3.254450,USA,"{'xmax': -117.23293286671338, 'xmin': -117.233...",Multi,1962.0,4.0,None,None,O,2.0,576122.0,living,2212.0,7998886,06073007702,h530,SDGE,RI101,113.884017,734.439912,POINT (-117.23297 32.79055),NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10012615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,2.0,None,None,O,NaN,1737925.0,living,2408.0,10714295,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26588 34.25417)
10012616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,921196.0,living,3301.0,10714296,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26591 34.25429)
10012617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,1503087.0,living,2667.0,10714297,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26594 34.25441)
10012618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Single,NaN,4.0,None,None,O,NaN,998162.0,living,3011.0,10714430,06111002500,h2540,Others,RR101,NaN,NaN,None,NaN,POINT (-119.26252 34.25562)


In [ ]:


# # join back to parcels
# units_multi = parcels_res.sjoin(multi_complete, predicate="intersects").groupby(level=0)['unit_right'].sum()

# # join on parcels with summed number of units
# multi_summed_units = parcels_res.join(units_multi)

# assert len(multi_summed_units) < len(multi_by_parcel)

# # save all non-multi observations
# non_multi = building_m[building_m['type'] != "Multi"].to_crs(zillow.crs)


# keep only variables of interest
# non_multi = non_multi[['source', 'id', 'height_m', 'var', 'region', 'bbox', 'geometry', 'area_m2', 'volume_m3']]

# join Zillow data to non-multi family homes (takes ~1 minute)
# non_multi_points = gpd.sjoin(
#     zillow, # left df's geometry is always kept
#     non_multi,
#     how = "inner",
#     predicate = "intersects")


## Final results

In [38]:
multi_complete.head() # should be my buildings??

,source,id,height_m,var,region,bbox,geometry,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
6345355,ms,UnitedStates_023013203_234397,3.895900,2.149426,USA,"{'xmax': -116.99674168516098, 'xmin': -116.997...","POLYGON ((-116.99693 33.29913, -116.99687 33.2...",Multi,2009.0,4.0,None,None,None,2.0,1359045.0,living,3704.0,7498446,06073019110,h221,SDGE,RI110,412.421351,1606.752247
6346533,ms,UnitedStates_023013203_158119,4.110258,0.494608,USA,"{'xmax': -116.98619956319268, 'xmin': -116.986...","POLYGON ((-116.98655 33.25268, -116.98620 33.2...",Multi,1985.0,5.0,None,None,O,2.0,585006.0,living,3630.0,7597631,06073019110,h464,SDGE,RI110,452.260448,1858.907171
6353683,osm,1136229920,3.897374,1.048893,USA,"{'xmax': -116.37151039999999, 'xmin': -116.371...","POLYGON ((-116.37161 33.25203, -116.37152 33.2...",Multi,1998.0,4.0,None,None,None,2.0,179623.0,living,1730.0,7599665,06073021002,h479,SDGE,RI101,232.343850,905.530915
6353687,osm,1136229917,4.132834,0.509725,USA,"{'xmax': -116.37210900000001, 'xmin': -116.372...","POLYGON ((-116.37224 33.25284, -116.37224 33.2...",Multi,1959.0,4.0,None,None,None,2.0,112047.0,living,1806.0,7599624,06073021002,h479,SDGE,RI101,284.844083,1177.213436
6353689,osm,1136229915,3.878115,0.254450,USA,"{'xmax': -116.37282159999998, 'xmin': -116.372...","POLYGON ((-116.37297 33.25286, -116.37283 33.2...",Multi,1962.0,4.0,None,None,None,2.0,155077.0,living,1736.0,7599626,06073021002,h479,SDGE,RI101,342.459460,1328.097228


In [8]:
multi_summed_units.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit,unit_right
23,4241720700,San Diego,None,None,None,98.936758,434.984710,"MULTIPOLYGON Z (((-117.23498 32.79751 0.00000,...",3.0,NaN
118,4241721300,San Diego,None,None,None,76.116160,347.815384,"MULTIPOLYGON Z (((-117.23439 32.79723 0.00000,...",2.0,NaN
119,4241722000,San Diego,None,None,None,106.557535,579.922629,"MULTIPOLYGON Z (((-117.23549 32.79698 0.00000,...",3.0,NaN


In [9]:
multi_by_parcel.head(3)

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,unit_left,index_right,...,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,area_m2,volume_m3
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,80603,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,130.144663,498.084000
151,4236921300,San Diego,None,None,None,69.352091,238.641330,"MULTIPOLYGON Z (((-117.25148 32.76739 0.00000,...",2.0,7349122,...,1026707.0,None,NaN,7996232.0,06073007602,h567,SDGE,RI101,140.235708,431.200368
154,4236220200,San Diego,None,None,None,67.240870,224.046738,"MULTIPOLYGON Z (((-117.25236 32.77776 0.00000,...",3.0,80751,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49.155171,171.388014


In [10]:
non_multi_points.head(3)

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,...,geometry,index_right,source,id,height_m,var,region,bbox,area_m2,volume_m3
7160203,Single,1959.0,3.0,None,None,None,1.0,179462.0,living,1780.0,...,POINT (-117.24993 33.38692),6701574,ms,UnitedStates_023013203_33032,3.835961,0.299546,USA,"{'xmin': -117.25014289023119, 'ymin': 33.38687...",301.592312,1156.896305
7160213,Single,1952.0,2.0,None,None,I,1.0,153443.0,living,1510.0,...,POINT (-117.24838 33.38564),6698354,ms,UnitedStates_023013203_96432,5.221156,0.925386,USA,"{'xmin': -117.24844943872387, 'ymin': 33.38562...",192.020400,1002.568394
7160303,Single,1962.0,3.0,None,None,I,2.0,401364.0,living,1540.0,...,POINT (-117.24780 33.38609),6698329,ms,UnitedStates_023013203_294280,6.511510,1.177574,USA,"{'xmin': -117.24791472273105, 'ymin': 33.38604...",237.514781,1546.579963


## Renaming and saving

In [13]:
multi_summed_units_sd = multi_summed_units.copy()

multi_by_parcel_sd = multi_by_parcel.copy()

non_multi_points_sd = non_multi_points.copy()

In [17]:
# takes a really long time :,(
multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 

In [15]:
non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

In [16]:
multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

In [14]:
## Saving! (takes at least 40 minutes)

multi_summed_units_sd.to_file("data/multi_summed_units_sd.geojson", driver='GeoJSON')

multi_by_parcel_sd.to_file("data/multi_by_parcel_sd.geojson", driver='GeoJSON')

non_multi_points_sd.to_file("data/non_multi_zillow_sd.geojson", driver='GeoJSON')

KeyboardInterrupt: 